# Student Addiction Project: MATH59854 methods in Python
Each section applies one course module to `student_addiction_dataset_test.csv`.
**In Colab:** run the first cell and upload the CSV when asked. **Locally:** put this notebook next to the CSV.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
FILE = 'student_addiction_dataset_test.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()          # choose student_addiction_dataset_test.csv
    except ImportError:
        raise FileNotFoundError('Put the CSV in the same folder as this notebook.')
raw = pd.read_csv(FILE)
factors = raw.columns[:-1].tolist()
df = raw.dropna().copy()          # same 7,516 complete rows as the Power BI report
yes = (df[factors] == 'Yes').astype(int)
df['risk_score'] = yes.sum(axis=1)
df['addicted'] = (df['Addiction_Class'] == 'Yes').astype(int)
print(f'Raw rows: {len(raw):,}   Complete rows: {len(df):,}')

## Module 2 · Data types & frequency distributions

In [ ]:
freq = df['risk_score'].value_counts().sort_index()
table = pd.DataFrame({'count': freq, 'percent': (freq/len(df)*100).round(1)})
table['cumulative_%'] = (freq.cumsum()/len(df)*100).round(1)
print(table)
print('\nMissing % per question (raw file):')
print((raw[factors].isna().mean()*100).round(1).sort_values(ascending=False))

## Module 3 · Measures of centre & spread

In [ ]:
s = df['risk_score']
summary = pd.Series({'mean': s.mean(), 'median': s.median(), 'mode': s.mode()[0],
                     'variance': s.var(), 'std dev': s.std(), 'IQR': s.quantile(.75)-s.quantile(.25),
                     'skewness': stats.skew(s)}).round(3)
print(summary)
print('\nBy addiction status:')
print(df.groupby('Addiction_Class')['risk_score'].agg(['mean','median','std','count']).round(3))

## Module 4 · Visualization

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
freq.plot(kind='bar', ax=ax[0], color='#426871')
ax[0].set(title='Students by risk score', xlabel='Risk score (warning signs)', ylabel='Students')
df.boxplot(column='risk_score', by='Addiction_Class', ax=ax[1], grid=False)
ax[1].set(title='Risk score by addiction status', xlabel='Addiction reported', ylabel='Risk score')
plt.suptitle(''); plt.tight_layout(); plt.show()

## Module 5 · Correlation & simple linear regression

In [ ]:
r = np.corrcoef(df['risk_score'], df['addicted'])[0, 1]
lr = stats.linregress(df['risk_score'], df['addicted'])
print(f'Correlation r = {r:.3f}')
print(f'Regression: addicted = {lr.intercept:.3f} + {lr.slope:.4f} x risk_score')
print(f'R-squared = {lr.rvalue**2:.4f}  (share of variation explained)   p = {lr.pvalue:.3f}')
print('\nCorrelation between the 10 factors (real behaviour would show clear positive values):')
corr = yes.corr().round(2)
print(corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max(), '= largest absolute correlation')

## Module 6 · Probability

In [ ]:
p_add = df['addicted'].mean()
p_risk = (df['Risk_Taking_Behavior']=='Yes').mean()
p_both = ((df['addicted']==1) & (df['Risk_Taking_Behavior']=='Yes')).mean()
print(f'P(addiction) = {p_add:.3f}')
print(f'P(addiction | risk-taking) = {p_both/p_risk:.3f}')
print(f'If independent, P(both) = {p_add*p_risk:.4f}; observed P(both) = {p_both:.4f}')

## Module 8 · Probability distributions (binomial)
If the 10 answers were independent coin flips with P(Yes) ≈ 0.31, the risk score would follow **Binomial(n=10, p≈0.31)**. Compare the observed counts with that model.

In [ ]:
p_yes = yes.values.mean()
k = np.arange(0, 11)
expected = stats.binom.pmf(k, 10, p_yes) * len(df)
comp = pd.DataFrame({'observed': freq.reindex(k, fill_value=0), 'binomial_expected': expected.round(0)})
print(f'p(Yes) = {p_yes:.3f}'); print(comp)
print(f'Observed mean/var: {s.mean():.2f} / {s.var():.2f}   Binomial mean/var: {10*p_yes:.2f} / {10*p_yes*(1-p_yes):.2f}')
comp.plot(kind='bar', color=['#426871', '#C25A3D'], figsize=(8,3.5), title='Observed vs binomial model'); plt.show()

## Module 9 · Normal distribution & the 68-95-99.7 rule

In [ ]:
m, sd = s.mean(), s.std()
for k_sd in (1, 2, 3):
    share = ((s >= m-k_sd*sd) & (s <= m+k_sd*sd)).mean()
    print(f'Within {k_sd} SD: {share:.1%}')
z = (s - m) / sd
print(f'Students with z > 2 (unusually high risk): {(z > 2).sum()}')

## Module 10 · Confidence intervals

In [ ]:
n = len(df)
se = np.sqrt(p_add*(1-p_add)/n)
print(f'Addiction rate: {p_add:.1%}, 95% CI {p_add-1.96*se:.1%} to {p_add+1.96*se:.1%}')
ci = stats.t.interval(0.95, n-1, loc=m, scale=stats.sem(s))
print(f'Mean risk score: {m:.2f}, 95% CI {ci[0]:.2f} to {ci[1]:.2f}')

## Module 11 · A/B-style comparison
Treat 'addiction reported' vs 'not reported' as groups A and B and test whether their mean risk scores differ.
(This dataset has no dates, so time-series forecasting needs a different dataset.)

In [ ]:
a = df.loc[df.addicted==1, 'risk_score']; b = df.loc[df.addicted==0, 'risk_score']
t = stats.ttest_ind(a, b, equal_var=False)
d = (a.mean()-b.mean()) / np.sqrt((a.var()+b.var())/2)
print(f'Mean A = {a.mean():.2f}, mean B = {b.mean():.2f}, t = {t.statistic:.2f}, p = {t.pvalue:.3f}, Cohen d = {d:.3f}')
print('Conclusion:', 'significant' if t.pvalue < .05 else 'not significant at 0.05', '- and the effect size is negligible.')